# Submission Performance Tables

This notebook rebuilds realized-game performance tables from saved frozen-submission artifact sets under `data/submissions/*_artifacts`.

It reports performance by:
- Season
- League
- Round
- Occurrence likelihood bucket

Round assignment uses the repository's canonical seed-pair lookup. Play-in games are excluded from the realized-game tables to match the repo's primary tournament evaluation policy.

In [7]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display

from mmlm2026.round_utils import assign_rounds_from_seeds

In [8]:
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = ROOT / "data" / "raw" / "march-machine-learning-mania-2026"
SUBMISSIONS_DIR = ROOT / "data" / "submissions"
ARTIFACT_GLOB = "*_artifacts"
ARTIFACT_TAG_FILTER: str | None = None

LEAGUE_CONFIG = {
    "M": {
        "prefix": "men",
        "results": DATA_DIR / "MNCAATourneyCompactResults.csv",
        "seeds": DATA_DIR / "MNCAATourneySeeds.csv",
    },
    "W": {
        "prefix": "women",
        "results": DATA_DIR / "WNCAATourneyCompactResults.csv",
        "seeds": DATA_DIR / "WNCAATourneySeeds.csv",
    },
}

bucket_order = ["definite", "very_likely", "likely", "plausible", "remote"]
round_group_order = ["R1", "R2+"]

artifact_dirs = sorted(SUBMISSIONS_DIR.glob(ARTIFACT_GLOB))
if ARTIFACT_TAG_FILTER:
    artifact_dirs = [path for path in artifact_dirs if ARTIFACT_TAG_FILTER in path.name]

artifact_dirs

[WindowsPath('C:/Users/brown/Documents/GitHub/mmlm2026/data/submissions/2025_gate3-dryrun-2025_artifacts'),
 WindowsPath('C:/Users/brown/Documents/GitHub/mmlm2026/data/submissions/2025_val-03-2025_artifacts')]

In [9]:
def parse_artifact_dir(artifact_dir: Path) -> tuple[int, str]:
    stem = artifact_dir.name.removesuffix("_artifacts")
    season_text, tag = stem.split("_", 1)
    return int(season_text), tag


def bucket_from_play_prob(play_prob: float) -> str:
    if play_prob >= 0.99:
        return "definite"
    if play_prob >= 0.30:
        return "very_likely"
    if play_prob >= 0.10:
        return "likely"
    if play_prob >= 0.03:
        return "plausible"
    return "remote"


def build_realized_results(league: str) -> pd.DataFrame:
    cfg = LEAGUE_CONFIG[league]
    results = pd.read_csv(cfg["results"])
    seeds = pd.read_csv(cfg["seeds"])
    realized = assign_rounds_from_seeds(results, seeds)
    realized = realized.loc[realized["Round"] > 0].copy()
    realized["LowTeamID"] = realized[["WTeamID", "LTeamID"]].min(axis=1)
    realized["HighTeamID"] = realized[["WTeamID", "LTeamID"]].max(axis=1)
    realized["ID"] = realized.apply(
        lambda row: f"{int(row['Season'])}_{int(row['LowTeamID'])}_{int(row['HighTeamID'])}",
        axis=1,
    )
    realized["outcome"] = (realized["WTeamID"] == realized["LowTeamID"]).astype(int)
    realized["actual_round"] = realized["Round"].astype(int)
    realized["actual_round_group"] = realized["actual_round"].map(
        lambda value: "R1" if value == 1 else "R2+"
    )
    return realized[
        ["Season", "ID", "LowTeamID", "HighTeamID", "outcome", "actual_round", "actual_round_group"]
    ]


REALIZED_RESULTS = {league: build_realized_results(league) for league in LEAGUE_CONFIG}


def load_artifact_performance(artifact_dir: Path, league: str) -> pd.DataFrame:
    season, tag = parse_artifact_dir(artifact_dir)
    cfg = LEAGUE_CONFIG[league]
    prefix = cfg["prefix"]

    predictions = pd.read_parquet(artifact_dir / f"{prefix}_predictions.parquet")
    play_probs = pd.read_csv(artifact_dir / f"{prefix}_bracket" / "play_probabilities.csv")
    play_probs["ID"] = play_probs.apply(
        lambda row: f"{int(row['Season'])}_{int(row['LowTeamID'])}_{int(row['HighTeamID'])}",
        axis=1,
    )
    play_probs["bucket"] = play_probs["play_prob"].map(bucket_from_play_prob)

    realized = REALIZED_RESULTS[league]
    season_realized = realized.loc[realized["Season"] == season].copy()
    merged = season_realized.merge(
        predictions[["ID", "Pred", "Round", "round_group"]],
        on="ID",
        how="left",
        validate="one_to_one",
    ).merge(
        play_probs[["ID", "play_prob", "bucket"]],
        on="ID",
        how="left",
        validate="one_to_one",
    )
    merged["artifact_tag"] = tag
    merged["league"] = league
    merged["predicted_round"] = merged["Round"]
    merged["predicted_round_group"] = merged["round_group"]
    merged["brier_component"] = (merged["Pred"] - merged["outcome"]).pow(2)
    return merged[
        [
            "artifact_tag",
            "Season",
            "league",
            "ID",
            "outcome",
            "Pred",
            "play_prob",
            "bucket",
            "actual_round",
            "actual_round_group",
            "predicted_round",
            "predicted_round_group",
            "brier_component",
        ]
    ]


frames = [
    load_artifact_performance(artifact_dir, league)
    for artifact_dir in artifact_dirs
    for league in LEAGUE_CONFIG
]
performance = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
performance.head()

,artifact_tag,Season,league,ID,outcome,Pred,play_prob,bucket,actual_round,actual_round_group,predicted_round,predicted_round_group,brier_component
0,gate3-dryrun-2025,2025,M,2025_1116_1242,1,0.313964,1.000000,definite,1,R1,1,R1,0.470645
1,gate3-dryrun-2025,2025,M,2025_1106_1120,0,0.005202,0.741856,very_likely,1,R1,1,R1,0.000027
2,gate3-dryrun-2025,2025,M,2025_1140_1433,1,0.628491,1.000000,definite,1,R1,1,R1,0.138019
3,gate3-dryrun-2025,2025,M,2025_1166_1257,1,0.540203,1.000000,definite,1,R1,1,R1,0.211413
4,gate3-dryrun-2025,2025,M,2025_1179_1281,1,0.445093,1.000000,definite,1,R1,1,R1,0.307922


In [10]:
def summarize(frame: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    if frame.empty:
        return frame
    summary = (
        frame.groupby(group_cols, dropna=False)
        .agg(
            games=("ID", "size"),
            mean_pred=("Pred", "mean"),
            mean_outcome=("outcome", "mean"),
            mean_play_prob=("play_prob", "mean"),
            flat_brier=("brier_component", "mean"),
        )
        .reset_index()
    )
    for col in ["mean_pred", "mean_outcome", "mean_play_prob", "flat_brier"]:
        summary[col] = summary[col].round(6)
    return summary.sort_values(group_cols).reset_index(drop=True)


combined_by_season = summarize(performance, ["artifact_tag", "Season"])
overview = summarize(performance, ["artifact_tag", "Season", "league"])
by_round = summarize(performance, ["artifact_tag", "Season", "league", "actual_round"])
by_round_group = summarize(performance, ["artifact_tag", "Season", "league", "actual_round_group"])
by_bucket = summarize(performance, ["artifact_tag", "Season", "league", "bucket"])
by_round_and_bucket = summarize(
    performance,
    ["artifact_tag", "Season", "league", "actual_round", "bucket"],
)

In [11]:
print("Combined Men + Women Overview by Season")
display(combined_by_season)

print("Overview by Season and League")
display(overview)

print("By Season, League, and Actual Round")
display(by_round)

print("By Season, League, and Actual Round Group")
display(by_round_group)

print("By Season, League, and Occurrence Likelihood Bucket")
display(
    by_bucket.assign(
        bucket=lambda df: pd.Categorical(df["bucket"], categories=bucket_order, ordered=True)
    ).sort_values(["artifact_tag", "Season", "league", "bucket"])
)

print("By Season, League, Actual Round, and Occurrence Likelihood Bucket")
display(
    by_round_and_bucket.assign(
        bucket=lambda df: pd.Categorical(df["bucket"], categories=bucket_order, ordered=True)
    ).sort_values(["artifact_tag", "Season", "league", "actual_round", "bucket"])
)

Combined Men + Women Overview by Season


,artifact_tag,Season,games,mean_pred,mean_outcome,mean_play_prob,flat_brier
0,gate3-dryrun-2025,2025,126,0.500345,0.507937,0.704666,0.120341
1,val-03-2025,2025,126,0.500345,0.507937,0.704666,0.120341


Overview by Season and League


,artifact_tag,Season,league,games,mean_pred,mean_outcome,mean_play_prob,flat_brier
0,gate3-dryrun-2025,2025,M,63,0.516612,0.587302,0.678347,0.134341
1,gate3-dryrun-2025,2025,W,63,0.484078,0.428571,0.730984,0.106341
2,val-03-2025,2025,M,63,0.516612,0.587302,0.678347,0.134341
3,val-03-2025,2025,W,63,0.484078,0.428571,0.730984,0.106341


By Season, League, and Actual Round


...

,artifact_tag,Season,league,actual_round_group,games,mean_pred,mean_outcome,mean_play_prob,flat_brier
0,gate3-dryrun-2025,2025,M,R1,32,0.533573,0.625000,0.940943,0.112829
1,gate3-dryrun-2025,2025,M,R2+,31,0.499103,0.548387,0.407281,0.156546
2,gate3-dryrun-2025,2025,W,R1,32,0.556941,0.500000,0.938715,0.085177
3,gate3-dryrun-2025,2025,W,R2+,31,0.408863,0.354839,0.516552,0.128187
4,val-03-2025,2025,M,R1,32,0.533573,0.625000,0.940943,0.112829
5,val-03-2025,2025,M,R2+,31,0.499103,0.548387,0.407281,0.156546
6,val-03-2025,2025,W,R1,32,0.556941,0.500000,0.938715,0.085177
7,val-03-2025,2025,W,R2+,31,0.408863,0.354839,0.516552,0.128187


By Season, League, and Occurrence Likelihood Bucket


,artifact_tag,Season,league,bucket,games,mean_pred,mean_outcome,mean_play_prob,flat_brier
0,gate3-dryrun-2025,2025,M,definite,28,0.532483,0.607143,1.000000,0.115327
4,gate3-dryrun-2025,2025,M,very_likely,25,0.505436,0.560000,0.519861,0.163090
1,gate3-dryrun-2025,2025,M,likely,7,0.535270,0.571429,0.221789,0.087977
2,gate3-dryrun-2025,2025,M,plausible,2,0.556681,1.000000,0.081284,0.260676
3,gate3-dryrun-2025,2025,M,remote,1,0.140869,0.000000,0.024244,0.019844
5,gate3-dryrun-2025,2025,W,definite,28,0.581868,0.535714,1.000000,0.092301
7,gate3-dryrun-2025,2025,W,very_likely,30,0.414332,0.333333,0.560571,0.112525
6,gate3-dryrun-2025,2025,W,likely,5,0.354921,0.400000,0.246972,0.147860
8,val-03-2025,2025,M,definite,28,0.532483,0.607143,1.000000,0.115327
12,val-03-2025,2025,M,very_likely,25,0.505436,0.560000,0.519861,0.163090


By Season, League, Actual Round, and Occurrence Likelihood Bucket


,artifact_tag,Season,league,actual_round,bucket,games,mean_pred,mean_outcome,mean_play_prob,flat_brier
0,gate3-dryrun-2025,2025,M,1,definite,28,0.532483,0.607143,1.000000,0.115327
1,gate3-dryrun-2025,2025,M,1,very_likely,4,0.541206,0.750000,0.527542,0.095343
3,gate3-dryrun-2025,2025,M,2,very_likely,15,0.473399,0.533333,0.568781,0.213346
2,gate3-dryrun-2025,2025,M,2,likely,1,0.120013,0.000000,0.150987,0.014403
7,gate3-dryrun-2025,2025,M,3,very_likely,3,0.603454,0.666667,0.462249,0.047200
4,gate3-dryrun-2025,2025,M,3,likely,3,0.590735,0.666667,0.252077,0.059996
5,gate3-dryrun-2025,2025,M,3,plausible,1,0.809950,1.000000,0.073400,0.036119
6,gate3-dryrun-2025,2025,M,3,remote,1,0.140869,0.000000,0.024244,0.019844
9,gate3-dryrun-2025,2025,M,4,very_likely,2,0.525630,0.500000,0.323856,0.047778
8,gate3-dryrun-2025,2025,M,4,likely,2,0.688770,1.000000,0.215107,0.096896


In [12]:
print("Pivoted flat Brier by Round and Bucket")
pivot = (
    by_round_and_bucket.assign(
        bucket=lambda df: pd.Categorical(df["bucket"], categories=bucket_order, ordered=True)
    )
    .pivot_table(
        index=["artifact_tag", "Season", "league", "actual_round"],
        columns="bucket",
        values="flat_brier",
        observed=False,
    )
    .reset_index()
)
display(pivot)

Pivoted flat Brier by Round and Bucket


C:\Users\brown\AppData\Local\Temp\ipykernel_141152\1705450276.py:6: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  .pivot_table(


bucket,artifact_tag,Season,league,actual_round,definite,very_likely,likely,plausible,remote
0,gate3-dryrun-2025,2025,M,1,0.115327,0.095343,NaN,NaN,NaN
1,gate3-dryrun-2025,2025,M,2,NaN,0.213346,0.014403,NaN,NaN
2,gate3-dryrun-2025,2025,M,3,NaN,0.047200,0.059996,0.036119,0.019844
3,gate3-dryrun-2025,2025,M,4,NaN,0.047778,0.096896,NaN,NaN
4,gate3-dryrun-2025,2025,M,5,NaN,0.258537,0.227656,NaN,NaN
5,gate3-dryrun-2025,2025,M,6,NaN,NaN,NaN,0.485233,NaN
6,gate3-dryrun-2025,2025,W,1,0.092301,0.035311,NaN,NaN,NaN
7,gate3-dryrun-2025,2025,W,2,NaN,0.126295,NaN,NaN,NaN
8,gate3-dryrun-2025,2025,W,3,NaN,0.110256,0.013757,NaN,NaN
9,gate3-dryrun-2025,2025,W,4,NaN,0.150656,0.108705,NaN,NaN
